In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
# surpress futre warnings
import warnings
import json
from utils import * 
warnings.simplefilter(action='ignore', category=FutureWarning)
import sys
import os
from pathlib import Path
# Add the src directory to the Python path
sys.path.append(os.path.abspath("../src"))
from guess_llm.probe.test import load_model_and_config

from scipy.stats import spearmanr
import scienceplots

sns.set_style('whitegrid')
plt.style.use('science')
plt.rcParams['font.family'] = 'sans-serif'

In [ ]:

save_dir = "" # path to the common saves directory
save_names = {
    'all' : "", # path to the dir with results of training on all real data
    'fold_0' : "", # path to the dir with results of training on fold 0
    'fold_1' : "", # path to the dir with results of training on fold 1
    'fold_2' : "", # path to the dir with results of training on fold 2
    'fold_3' : "", # path to the dir with results of training on fold 3
    'fold_4' : "", # path to the dir with results of training on fold 4
}
synth_results_path = "" # path to the csv with results of training on synthetic data

results_df = pd.DataFrame()

for train_type, save_path in save_names.items():
    # Load the data
    test_df = pd.read_csv(f'{save_dir}/{save_path}/test_results.csv')
    test_df['eval_type'] = 'test'
    test_df['train_type'] = train_type
    results_df = pd.concat([results_df, test_df], ignore_index=True)

synthetic_results_df = pd.read_csv(f'{save_dir}/{synth_results_path}')
synthetic_results_df['eval_type'] = 'test'
synthetic_results_df['train_type'] = 'Synth'

results_df = pd.concat([results_df, synthetic_results_df], ignore_index=True).reset_index(drop=True)

results_df['y'] = results_df['y'].apply(lambda x: json.loads(x))
results_df['pred_quantiles'] = results_df['pred_quantiles'].apply(lambda x: json.loads(x))
results_df['pred_exp_quantiles'] = results_df['pred_exp_quantiles'].apply(lambda x: json.loads(x))
results_df['train'] = results_df['train'].apply(lambda x: json.loads(x))

_, config = load_model_and_config(save_dir + '/' + save_names['all'])
QUANTILE_LIST = config.model.quantiles
EPSILON = 1e-3

In [ ]:
results_df['train_type'] = ['Real (5 fold)' if 'fold' in x else x for x in results_df['train_type']]
results_df['train_type'] = ['Real (all)' if x == 'all' else x for x in results_df['train_type']]
results_df['sample_median'] = results_df['y'].apply(lambda x: np.median(x))
results_df['pred_median'] = results_df['pred_exp_quantiles'].apply(lambda x: x[QUANTILE_LIST.index(0.5)])
results_df['median_abs_error'] = (results_df['sample_median'] - results_df['pred_median']).abs()
results_df['median_mse_error'] = (results_df['sample_median'] - results_df['pred_median'])**2
results_df['rel_median_abs_error'] = (results_df['sample_median'] - results_df['pred_median']).abs() / (results_df['sample_median'].abs() + EPSILON)

# compute median of y per dataset
mean_median = results_df.groupby('dataset')['sample_median'].median().reset_index()
mean_median.columns = ['dataset', 'median_dataset_y_median']
mean_median['median_dataset_y_median_str'] = mean_median.apply(lambda x: f"{x['dataset']} ({x['median_dataset_y_median']:.1e})", axis=1)
results_df = pd.merge(
    results_df,
    mean_median,
    how='left',
    on='dataset',
)

In [ ]:
plot_df = results_df.copy()

# set as categorical

plot_df['train_type'] = pd.Categorical(
    plot_df['train_type'],
    categories=['Real (all)', 'Real (5 fold)', 'Synth'],
    ordered=True
)

fig, ax = plt.subplots(figsize=(13, 3))
sns.boxplot(
    data=plot_df.sort_values('median_dataset_y_median'),
    x='median_dataset_y_median_str', y='median_abs_error', 
    hue='train_type', legend=True, showfliers=False, linewidth=0.8,
)
ax.set_yscale('log')
# rotate text on x axis
plt.xticks(rotation=40, ha='right')
ax.set_ylabel('Absolute Error on the Median')
ax.set_xlabel('Sub-Dataset (Avg. Median of LLM predictions)')
ax.legend(title='Training Type')
plt.show()

In [ ]:
results_df['95_coverage'] = results_df.apply(lambda x: get_coverage_intervals(x['y'], x['pred_exp_quantiles'], QUANTILE_LIST, ci=95), axis=1)
results_df['90_coverage'] = results_df.apply(lambda x: get_coverage_intervals(x['y'], x['pred_exp_quantiles'], QUANTILE_LIST, ci=90), axis=1)
results_df['50_coverage'] = results_df.apply(lambda x: get_coverage_intervals(x['y'], x['pred_exp_quantiles'], QUANTILE_LIST, ci=50), axis=1)

In [ ]:
print_df = results_df.copy()
print_df = print_df.groupby(['train_type'])[['95_coverage', '90_coverage', '50_coverage']].aggregate(['mean', 'sem']).unstack().reset_index()
print_df.columns = ['ci', 'key', 'train_type', 'value']
print_df = print_df.pivot(index=['ci', 'train_type'], columns='key', values='value').reset_index()
print_df['print_value'] = print_df.apply(lambda x: f"{x['mean'] * 100:.1f} ± {x['sem'] * 100:.1f}", axis=1)
print_df = print_df.pivot(index='train_type', columns=['ci'], values='print_value').reset_index()
print_df

In [ ]:
print(print_df.to_latex(index=False))